In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
import ipywidgets as widgets
%matplotlib inline

In [2]:
# Black-Scholes Delta Function
def calculate_delta(S, K, T, r, sigma, option_type, position):
    # Avoid division by zero for T=0
    T = max(T, 0.0001) 
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    
    if option_type == 'Call':
        delta = norm.cdf(d1)
    else: # Put
        delta = norm.cdf(d1) - 1
        
    return delta * (1 if position == 'Long' else -1)

def update_plots_delta(S_init, K, sigma, days_to_expiry, r, option_type, position):
    T = days_to_expiry / 365
    r_decimal = r / 100
    sigma_decimal = sigma / 100
    
    # Generate ranges for the X-axes
    s_range = np.linspace(K * 0.5, K * 1.5, 100)
    t_range = np.linspace(0.001, 2, 100) # up to 2 years
    vol_range = np.linspace(0.05, 1.0, 100) # 5% to 100%
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))
    plt.subplots_adjust(hspace=0.4)

    # 1. Delta vs Underlying Price
    deltas_s = [calculate_delta(s, K, T, r_decimal, sigma_decimal, option_type, position) for s in s_range]
    ax1.plot(s_range, deltas_s, color='blue')
    ax1.axvline(S_init, color='red', linestyle='--', label=f'Current S: {S_init}')
    ax1.set_title("Delta vs Underlying Price")
    ax1.set_xlabel("Stock Price")
    ax1.legend()

    # 2. Delta vs Days to Expiry
    deltas_t = [calculate_delta(S_init, K, t, r_decimal, sigma_decimal, option_type, position) for t in t_range]
    ax2.plot(t_range * 365, deltas_t, color='green')
    ax2.set_title("Delta vs Days to Expiry")
    ax2.set_xlabel("Days")

    # 3. Delta vs Volatility
    deltas_v = [calculate_delta(S_init, K, T, r_decimal, v, option_type, position) for v in vol_range]
    ax3.plot(vol_range * 100, deltas_v, color='purple')
    ax3.set_title("Delta vs Volatility")
    ax3.set_xlabel("Volatility (%)")

    # 4. Delta vs Strike Price
    k_range = np.linspace(S_init * 0.5, S_init * 1.5, 100)
    deltas_k = [calculate_delta(S_init, k, T, r_decimal, sigma_decimal, option_type, position) for k in k_range]
    ax4.plot(k_range, deltas_k, color='orange')
    ax4.set_title("Delta vs Strike Price")
    ax4.set_xlabel("Strike")

    for ax in [ax1, ax2, ax3, ax4]:
        ax.grid(True, alpha=0.3)
        ax.set_ylabel("Delta")

    plt.show()

# Create interactive sliders
interact(update_plots_delta, 
    S_init=IntSlider(value=100, min=10, max=500, step=1, description='Stock Price'),
    K=IntSlider(value=100, min=10, max=500, step=1, description='Strike'),
    sigma=FloatSlider(value=25, min=1, max=100, step=1, description='Vol %'),
    days_to_expiry=IntSlider(value=30, min=1, max=365, step=1, description='Days Left'),
    r=FloatSlider(value=5.0, min=0, max=15, step=0.1, description='Rate %'),
    option_type=Dropdown(options=['Call', 'Put'], value='Call', description='Type'),
    position=Dropdown(options=['Long', 'Short'], value='Long', description='Position')
);

interactive(children=(IntSlider(value=100, description='Stock Price', max=500, min=10), IntSlider(value=100, d…

In [3]:
def calculate_gamma(S, K, T, r, sigma, position):
    # Avoid division by zero
    T = max(T, 0.0001) 
    sigma = max(sigma, 0.0001)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    
    # Gamma Formula: PDF(d1) / (S * vol * sqrt(T))
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    
    return gamma * (1 if position == 'Long' else -1)

def update_gamma_plots(S_init, K, sigma, days_to_expiry, r, position):
    T = days_to_expiry / 365
    r_decimal = r / 100
    sigma_decimal = sigma / 100
    
    # Ranges for X-axes
    s_range = np.linspace(K * 0.5, K * 1.5, 100)
    t_range = np.linspace(1, 365, 100) / 365 # 1 day to 1 year
    vol_range = np.linspace(0.05, 1.0, 100) # 5% to 100%
    k_range = np.linspace(S_init * 0.5, S_init * 1.5, 100)
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
    plt.subplots_adjust(hspace=0.3, wspace=0.3)

    # 1. Gamma vs Underlying Price (The "Bell Curve")
    gammas_s = [calculate_gamma(s, K, T, r_decimal, sigma_decimal, position) for s in s_range]
    ax1.plot(s_range, gammas_s, color='darkorange', lw=2)
    ax1.axvline(S_init, color='red', linestyle='--', label=f'Current S: {S_init}')
    ax1.set_title("Gamma vs Underlying Price")
    ax1.set_xlabel("Stock Price")
    ax1.legend()

    # 2. Gamma vs Days to Expiry
    gammas_t = [calculate_gamma(S_init, K, t, r_decimal, sigma_decimal, position) for t in t_range]
    ax2.plot(t_range * 365, gammas_t, color='teal', lw=2)
    ax2.set_title("Gamma vs Days to Expiry")
    ax2.set_xlabel("Days to Expiration")

    # 3. Gamma vs Volatility
    gammas_v = [calculate_gamma(S_init, K, T, r_decimal, v, position) for v in vol_range]
    ax3.plot(vol_range * 100, gammas_v, color='crimson', lw=2)
    ax3.set_title("Gamma vs Volatility")
    ax3.set_xlabel("Volatility (%)")

    # 4. Gamma vs Strike Price
    gammas_k = [calculate_gamma(S_init, k, T, r_decimal, sigma_decimal, position) for k in k_range]
    ax4.plot(k_range, gammas_k, color='navy', lw=2)
    ax4.set_title("Gamma vs Strike Price")
    ax4.set_xlabel("Strike")

    for ax in [ax1, ax2, ax3, ax4]:
        ax.grid(True, alpha=0.3)
        ax.set_ylabel("Gamma")

    plt.suptitle(f"{position} Position Gamma Analysis", fontsize=16)
    plt.show()

interact(update_gamma_plots, 
    S_init=IntSlider(value=100, min=10, max=500, step=1, description='Stock Price'),
    K=IntSlider(value=100, min=10, max=500, step=1, description='Strike'),
    sigma=FloatSlider(value=25, min=1, max=100, step=1, description='Vol %'),
    days_to_expiry=IntSlider(value=30, min=1, max=365, step=1, description='Days Left'),
    r=FloatSlider(value=5.0, min=0, max=15, step=0.1, description='Rate %'),
    position=Dropdown(options=['Long', 'Short'], value='Long')
);

interactive(children=(IntSlider(value=100, description='Stock Price', max=500, min=10), IntSlider(value=100, d…

In [4]:
def calculate_deltagamma(S, K, T, r, sigma, option_type, position):
    T = max(T, 0.0001) 
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    
    # Delta Calculation
    if option_type == 'Call':
        delta = norm.cdf(d1)
    else:
        delta = norm.cdf(d1) - 1
    
    # Gamma Calculation (Same for Call and Put)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    
    multiplier = 1 if position == 'Long' else -1
    return delta * multiplier, gamma * multiplier

def update_deltagamma_plots(S_init, K, sigma, days_to_expiry, r, option_type, position):
    T = days_to_expiry / 365
    r_dec, sig_dec = r / 100, sigma / 100
    s_range = np.linspace(K * 0.5, K * 1.5, 100)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Calculate both Greeks across the stock price range
    greeks = [calculate_deltagamma(s, K, T, r_dec, sig_dec, option_type, position) for s in s_range]
    deltas, gammas = zip(*greeks)

    # Plot Delta
    ax1.plot(s_range, deltas, color='blue', lw=2)
    ax1.axvline(S_init, color='red', linestyle='--', label=f'Current S: {S_init}')
    ax1.set_title(f"{position} {option_type} Delta")
    ax1.set_ylabel("Delta")
    ax1.grid(True, alpha=0.3)

    # Plot Gamma
    ax2.plot(s_range, gammas, color='orange', lw=2)
    ax2.axvline(S_init, color='red', linestyle='--')
    ax2.set_title(f"{position} {option_type} Gamma")
    ax2.set_ylabel("Gamma")
    ax2.grid(True, alpha=0.3)

    plt.show()

interact(update_deltagamma_plots, 
    S_init=IntSlider(value=100, min=10, max=500, step=1),
    K=IntSlider(value=100, min=10, max=500, step=1),
    sigma=FloatSlider(value=25, min=1, max=100, step=1),
    days_to_expiry=IntSlider(value=30, min=1, max=365, step=1),
    r=FloatSlider(value=5.0, min=0, max=15, step=0.1),
    option_type=['Call', 'Put'],
    position=['Long', 'Short']
);

interactive(children=(IntSlider(value=100, description='S_init', max=500, min=10), IntSlider(value=100, descri…

In [5]:
def calculate_theta(S, K, T, r, sigma, option_type, position):
    T = max(T, 0.0001)
    sigma = max(sigma, 0.0001)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    term1 = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
    
    if option_type == 'Call':
        term2 = r * K * np.exp(-r * T) * norm.cdf(d2)
        theta = term1 - term2
    else: # Put
        term2 = r * K * np.exp(-r * T) * norm.cdf(-d2)
        theta = term1 + term2
        
    # Return daily theta
    return (theta / 365) * (1 if position == 'Long' else -1)

def update_theta_plots(S_init, K, sigma, days_to_expiry, r, option_type, position):
    T = days_to_expiry / 365
    r_dec, sig_dec = r / 100, sigma / 100
    
    s_range = np.linspace(K * 0.5, K * 1.5, 100)
    t_range = np.linspace(1, 365, 100) / 365
    vol_range = np.linspace(0.05, 1.0, 100)
    k_range = np.linspace(S_init * 0.5, S_init * 1.5, 100)
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
    plt.subplots_adjust(hspace=0.3, wspace=0.3)

    # 1. Theta vs Underlying Price
    thetas_s = [calculate_theta(s, K, T, r_dec, sig_dec, option_type, position) for s in s_range]
    ax1.plot(s_range, thetas_s, color='forestgreen', lw=2)
    ax1.axvline(S_init, color='red', linestyle='--')
    ax1.set_title("Daily Theta vs Underlying Price")
    ax1.set_xlabel("Stock Price")

    # 2. Theta vs Days to Expiry (The acceleration of decay)
    thetas_t = [calculate_theta(S_init, K, t, r_dec, sig_dec, option_type, position) for t in t_range]
    ax2.plot(t_range * 365, thetas_t, color='blue', lw=2)
    ax2.set_title("Daily Theta vs Days to Expiry")
    ax2.set_xlabel("Days to Expiration")
    ax2.invert_xaxis() # Show decay moving toward 0 days

    # 3. Theta vs Volatility
    thetas_v = [calculate_theta(S_init, K, T, r_dec, v, option_type, position) for v in vol_range]
    ax3.plot(vol_range * 100, thetas_v, color='tab:red', lw=2)
    ax3.set_title("Daily Theta vs Volatility")
    ax3.set_xlabel("Volatility (%)")

    # 4. Theta vs Strike Price
    thetas_k = [calculate_theta(S_init, k, T, r_dec, sig_dec, option_type, position) for k in k_range]
    ax4.plot(k_range, thetas_k, color='dimgray', lw=2)
    ax4.set_title("Daily Theta vs Strike Price")
    ax4.set_xlabel("Strike")

    for ax in [ax1, ax2, ax3, ax4]:
        ax.grid(True, alpha=0.3)
        ax.set_ylabel("Daily Theta ($)")

    plt.suptitle(f"{position} {option_type} Theta Analysis", fontsize=16)
    plt.show()

interact(update_theta_plots, 
    S_init=IntSlider(value=100, min=10, max=500, step=1),
    K=IntSlider(value=100, min=10, max=500, step=1),
    sigma=FloatSlider(value=25, min=1, max=100, step=1),
    days_to_expiry=IntSlider(value=30, min=1, max=365, step=1),
    r=FloatSlider(value=5.0, min=0, max=15, step=0.1),
    option_type=['Call', 'Put'],
    position=['Long', 'Short']
);

interactive(children=(IntSlider(value=100, description='S_init', max=500, min=10), IntSlider(value=100, descri…

In [6]:
def calculate_gammatheta(S, K, T, r, sigma):
    T = max(T, 0.0001)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # Gamma (Same for Call/Put)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    
    # Theta (Using Call Theta for the visualization)
    term1 = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
    term2 = r * K * np.exp(-r * T) * norm.cdf(d2)
    theta_daily = (term1 - term2) / 365
    
    return gamma, theta_daily

def plot_tradeoff(K, sigma, days_to_expiry, r):
    S_range = np.linspace(K * 0.7, K * 1.3, 150)
    T = days_to_expiry / 365
    sig_dec = sigma / 100
    r_dec = r / 100
    
    gammas = []
    thetas = []
    
    for s in S_range:
        g, t = calculate_gammatheta(s, K, T, r_dec, sig_dec)
        gammas.append(g)
        thetas.append(t)

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Plot Gamma on the left axis
    color_gamma = 'tab:orange'
    ax1.set_xlabel('Stock Price')
    ax1.set_ylabel('Gamma (Sensitivity)', color=color_gamma, fontsize=12)
    ax1.plot(S_range, gammas, color=color_gamma, lw=3, label='Gamma (The Benefit)')
    ax1.tick_params(axis='y', labelcolor=color_gamma)
    ax1.fill_between(S_range, gammas, alpha=0.1, color=color_gamma)

    # Create a second axis for Theta
    ax2 = ax1.twinx()
    color_theta = 'tab:red'
    ax2.set_ylabel('Daily Theta (The Cost)', color=color_theta, fontsize=12)
    ax2.plot(S_range, thetas, color=color_theta, lw=3, label='Theta (The Rent)')
    ax2.tick_params(axis='y', labelcolor=color_theta)
    
    # Add a horizontal line at 0 for Theta
    ax2.axhline(0, color='black', lw=1)

    plt.title(f'The Gamma-Theta Trade-off\n(High Gamma always costs High Theta)', fontsize=14)
    ax1.grid(True, alpha=0.3)
    
    # Identify the Strike
    ax1.axvline(K, color='gray', linestyle='--', label=f'Strike: {K}')
    
    fig.tight_layout()
    plt.show()

interact(plot_tradeoff, 
    K=IntSlider(value=100, min=50, max=150),
    sigma=FloatSlider(value=25, min=5, max=100),
    days_to_expiry=IntSlider(value=30, min=1, max=180),
    r=FloatSlider(value=5.0, min=0, max=15)
);

interactive(children=(IntSlider(value=100, description='K', max=150, min=50), FloatSlider(value=25.0, descript…

In [7]:
def calculate_vega(S, K, T, r, sigma, position):
    T = max(T, 0.0001)
    sigma = max(sigma, 0.0001)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    
    # Vega Formula: S * PDF(d1) * sqrt(T)
    # Divided by 100 to show the price change for a 1% move in Vol
    vega = (S * norm.pdf(d1) * np.sqrt(T)) / 100
    
    return vega * (1 if position == 'Long' else -1)

def update_vega_plots(S_init, K, sigma, days_to_expiry, r, position):
    T = days_to_expiry / 365
    r_dec, sig_dec = r / 100, sigma / 100
    
    s_range = np.linspace(K * 0.5, K * 1.5, 100)
    t_range = np.linspace(1, 365, 100) / 365
    vol_range = np.linspace(0.05, 1.0, 100)
    k_range = np.linspace(S_init * 0.5, S_init * 1.5, 100)
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
    plt.subplots_adjust(hspace=0.3, wspace=0.3)

    # 1. Vega vs Underlying Price
    vegas_s = [calculate_vega(s, K, T, r_dec, sig_dec, position) for s in s_range]
    ax1.plot(s_range, vegas_s, color='mediumorchid', lw=2)
    ax1.axvline(S_init, color='red', linestyle='--')
    ax1.set_title("Vega vs Underlying Price")
    ax1.set_xlabel("Stock Price")

    # 2. Vega vs Days to Expiry
    vegas_t = [calculate_vega(S_init, K, t, r_dec, sig_dec, position) for t in t_range]
    ax2.plot(t_range * 365, vegas_t, color='royalblue', lw=2)
    ax2.set_title("Vega vs Days to Expiry")
    ax2.set_xlabel("Days to Expiration")

    # 3. Vega vs Volatility
    vegas_v = [calculate_vega(S_init, K, T, r_dec, v, position) for v in vol_range]
    ax3.plot(vol_range * 100, vegas_v, color='darkorange', lw=2)
    ax3.set_title("Vega vs Volatility")
    ax3.set_xlabel("Volatility (%)")

    # 4. Vega vs Strike Price
    vegas_k = [calculate_vega(S_init, k, T, r_dec, sig_dec, position) for k in k_range]
    ax4.plot(k_range, vegas_k, color='seagreen', lw=2)
    ax4.set_title("Vega vs Strike Price")
    ax4.set_xlabel("Strike")

    for ax in [ax1, ax2, ax3, ax4]:
        ax.grid(True, alpha=0.3)
        ax.set_ylabel("Vega ($ per 1% Vol)")

    plt.suptitle(f"{position} Position Vega Analysis", fontsize=16)
    plt.show()

interact(update_vega_plots, 
    S_init=IntSlider(value=100, min=10, max=500, step=1),
    K=IntSlider(value=100, min=10, max=500, step=1),
    sigma=FloatSlider(value=25, min=1, max=100, step=1),
    days_to_expiry=IntSlider(value=30, min=1, max=365, step=1),
    r=FloatSlider(value=5.0, min=0, max=15, step=0.1),
    position=['Long', 'Short']
);

interactive(children=(IntSlider(value=100, description='S_init', max=500, min=10), IntSlider(value=100, descri…